In [1]:
!nvidia-smi
!pip install -q ultralytics

Thu Oct  8 12:50:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_NAME = "dataset.zip"   # Drive mein jo aapki zip ka naam hai wahi likho

!cp "/content/drive/MyDrive/{ZIP_NAME}" /content/data.zip
!rm -rf /content/ds
!unzip -q -o /content/data.zip -d /content/ds

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import os

root = None
for dirpath, dirnames, _ in os.walk("/content/ds"):
    if "train" in dirnames and "valid" in dirnames:
        root = dirpath
        break
print("Dataset root:", root)
print(os.listdir(root))

yaml_text = f"""path: {root}
train: train/images
val: valid/images
test: test/images

nc: 2
names: ['With Helmet', 'Without Helmet']
"""
with open("/content/data.yaml", "w") as f:
    f.write(yaml_text)
print(yaml_text)

Dataset root: /content/ds/dataset
['data.yaml', 'test', 'Helmet_detect.ipynb', 'train', 'valid', 'README.dataset.txt', 'README.roboflow.txt']
path: /content/ds/dataset
train: train/images
val: valid/images
test: test/images

nc: 2
names: ['With Helmet', 'Without Helmet']



In [21]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="/content/data.yaml",
    epochs=100,
    patience=20,
    imgsz=600,
    batch=16,
    name="baseline_n",
    project="/content/drive/MyDrive/ppe_runs",
)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=600, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_n-5, nbs=64, nms=None, opset=None, optimize=False, optimizer=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7945d968d0f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [22]:
from ultralytics import YOLO
import glob

best = glob.glob("/content/drive/MyDrive/ppe_runs/**/baseline_n*/weights/best.pt", recursive=True)[0]
print("Using:", best)

model = YOLO(best)
m = model.val(data="/content/data.yaml", split="test")
print("mAP50:", m.box.map50, "| mAP50-95:", m.box.map)
print("Precision:", m.box.mp, "| Recall:", m.box.mr)

Using: /content/drive/MyDrive/ppe_runs/baseline_n/weights/best.pt
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1208.3±323.9 MB/s, size: 45.9 KB)
val: Scanning /content/ds/dataset/test/labels.cache... 128 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 28.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.7it/s 4.7s
                   all        128        231      0.669      0.751      0.734      0.436
           With Helmet         98        177      0.838      0.816       0.85      0.547
        Without Helmet         33         54        0.5      0.686      0.619      0.324
Speed: 7.9ms preprocess, 6.1ms inference, 0.0ms loss, 5.5ms postprocess per image
Results saved to /content/runs/detect/val-3
mAP50: 0.734322980625853 | mAP50-9